# Going to see the effect of Normalization on the model.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

In [ ]:
# Employee Performance Score Prediction

torch.manual_seed(42)
samples = 1000
weights = torch.Tensor([0.5, 0.9, 0.3, 0.4, 0.8]).unsqueeze(1)
bias = 0.4

attendance = torch.randint(60, 101, (samples, 1))          # Percentage # range(60 - 100)
experience = torch.randint(0, 11, (samples, 1))            # Years # range (0 - 10)
current_projects = torch.randint(1, 6, (samples, 1))       # range (1 - 5)
completed_projects = (experience  * 2) + torch.randint(1, 35, (samples, 1)) # range (1 - 34)
weekly_working_hours = torch.randint(30, 51, (samples, 1)) # range (30 - 50)

X = torch.concat((attendance, experience, current_projects, completed_projects, weekly_working_hours), dim=1).float()
X = (X - X.min()) / (X.max() - X.min()) 
noise = torch.randn(samples, 1) * 5  # noise level
y = torch.matmul(X, weights) + bias # + noise if you uncomment this line this current model accuracy will go down because of the current approach of loss function and optimzer we had used # Employee Performance Score
y = (y - y.min()) / (y.max() - y.min()) * 100
X[:10] , y[:10]
# ranges
# Attendance         : 60–100
# Experience         : 0–10
# Current Projects   : 1–5
# Completed Projects : 1–55
# Weekly Hours       : 30–50

In [ ]:
split_ratio = int(0.8 * len(X))

x_train, y_train = X[:split_ratio], y[:split_ratio]
x_test, y_test = X[split_ratio:], y[split_ratio:]

len(x_train), len(y_train), len(x_test), len(y_test)

In [ ]:
feature_names = [
    "Attendance (%)",
    "Experience (Years)",
    "Current Projects",
    "Completed Projects",
    "Weekly Working Hours"
]

def dataset_intuition(
    feature_index=0,
    x_train=x_train,
    y_train=y_train,
    x_test=x_test,
    y_test=y_test,
    predictions=None
):
    plt.figure(figsize=(18, 16))

    plt.scatter(
        x_train[:, feature_index],
        y_train,
        color="green",
        s=20,
        label="Training Data"
    )

    plt.scatter(
        x_test[:, feature_index],
        y_test,
        color="blue",
        s=35,
        label="Testing Data"
    )

    if predictions is not None:
        plt.scatter(
            x_test[:, feature_index],
            predictions,
            color="red",
            s=20,
            label="Predictions"
        )

    plt.xlabel(feature_names[feature_index], fontsize=12)
    plt.ylabel("Employee Performance Score", fontsize=12)
    plt.title(f"{feature_names[feature_index]} vs Performance Score", fontsize=14)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
    
dataset_intuition(4)

In [ ]:
class PerformanceScorePredictorModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(5,1), requires_grad=True)        
        self.bias = nn.Parameter(torch.rand(1), requires_grad=True)        
        
    def forward(self, X:torch.Tensor):
        return torch.matmul(X, self.weights) + self.bias

In [ ]:
torch.manual_seed(42)
model = PerformanceScorePredictorModel()
print(model.state_dict())

In [ ]:
with torch.inference_mode():
    ypred = model(x_test)
ypred
dataset_intuition(feature_index=4,predictions=ypred)

In [ ]:
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.1)

In [ ]:
epochs = 1000

for epoch in range(epochs):
    model.train()
    
    model_predictions = model(x_train)
    train_loss = loss_fn(model_predictions,y_train)
    
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    model.eval()
    
    if epoch % 10 == 0:                        
        print(f"Epoch number: {epoch + 1 } | MAE Train Loss: {train_loss} ")
        
        
    

In [ ]:
with torch.inference_mode():
    test_pred = model(x_test)
print(model.state_dict())
print(f"Actual weights: " , weights)
print(f"Actual bias: " , bias)
print(test_pred)
dataset_intuition(feature_index=0,predictions=test_pred)